In [ ]:
# !pip install rectools[lightfm] matplotlib seaborn polars

In [422]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import rectools
from rectools import Columns

In [423]:
# Считываем датасеты

df_users = pd.read_csv('data_original/data_original/users.csv')
df_interactions = pd.read_csv('data_original/data_original/interactions.csv')
df_items = pd.read_csv('data_original/data_original/items.csv')

In [424]:
df_users = df_users.drop(df_users.index[-1])
df_interactions = df_interactions.drop(df_interactions.index[-1])
df_items = df_items.drop(df_items.index[-1])

In [425]:
df_items.head()

,item_id,content_type,title,title_orig,release_year,genres,countries,for_kids,age_rating,studios,directors,actors,description,keywords
0,10711,film,Поговори с ней,Hable con ella,2002.0,"драмы, зарубежные, детективы, мелодрамы",Испания,NaN,16.0,NaN,Педро Альмодовар,"Адольфо Фернандес, Ана Фернандес, Дарио Гранди...",Мелодрама легендарного Педро Альмодовара «Пого...,"Поговори, ней, 2002, Испания, друзья, любовь, ..."
1,2508,film,Голые перцы,Search Party,2014.0,"зарубежные, приключения, комедии",США,NaN,16.0,NaN,Скот Армстронг,"Адам Палли, Брайан Хаски, Дж.Б. Смув, Джейсон ...",Уморительная современная комедия на популярную...,"Голые, перцы, 2014, США, друзья, свадьбы, прео..."
2,10716,film,Тактическая сила,Tactical Force,2011.0,"криминал, зарубежные, триллеры, боевики, комедии",Канада,NaN,16.0,NaN,Адам П. Калтраро,"Адриан Холмс, Даррен Шалави, Джерри Вассерман,...",Профессиональный рестлер Стив Остин («Все или ...,"Тактическая, сила, 2011, Канада, бандиты, ганг..."
3,7868,film,45 лет,45 Years,2015.0,"драмы, зарубежные, мелодрамы",Великобритания,NaN,16.0,NaN,Эндрю Хэй,"Александра Риддлстон-Барретт, Джеральдин Джейм...","Шарлотта Рэмплинг, Том Кортни, Джеральдин Джей...","45, лет, 2015, Великобритания, брак, жизнь, лю..."
4,16268,film,Все решает мгновение,NaN,1978.0,"драмы, спорт, советские, мелодрамы",СССР,NaN,12.0,Ленфильм,Виктор Садовский,"Александр Абдулов, Александр Демьяненко, Алекс...",Расчетливая чаровница из советского кинохита «...,"Все, решает, мгновение, 1978, СССР, сильные, ж..."


In [426]:
df_interactions['last_watch_dt'] = pd.to_datetime(df_interactions['last_watch_dt'], errors='coerce')

In [427]:
# def check_date_with_formats(df, column_name, date_formats=None):
#     """
#     Проверка дат с учетом различных форматов
#     """
#     if date_formats is None:
#         date_formats = ['%Y-%m-%d', '%d.%m.%Y', '%m/%d/%Y', '%Y-%m-%d %H:%M:%S']
    
#     results = {}
    
#     for date_format in date_formats:
#         try:
#             # Пробуем преобразовать с конкретным форматом
#             converted = pd.to_datetime(df[column_name], format=date_format, errors='coerce')
#             valid_count = converted.notna().sum()
#             results[date_format] = valid_count
#         except Exception:
#             results[date_format] = 0
    
#     # Находим лучший формат
#     best_format = max(results.items(), key=lambda x: x[1])
    
#     return {
#         'best_format': best_format[0] if best_format[1] > 0 else None,
#         'valid_count_with_best_format': best_format[1],
#         'all_results': results
#     }

In [428]:
df_interactions['completed'] = (df_interactions['watched_pct'] >= 60).astype(int)

In [429]:
# def convert_to_datetime(series):
#     result = pd.Series(index=series.index)
#     idx1 = ['-' in i if i is not None else True  for i in series] # Ищет формат даты через "-" и None
#     idx2 = [not i for i in idx1] # Делает замену True на False и False на True
#     result[idx1] = pd.to_datetime(series[idx1], format='%Y-%m-%d')
#     result[idx2] = pd.to_datetime(series[idx2], format='%d %m %Y')
    
#     return result

In [430]:
df_inter = df_interactions.drop(['total_dur'], axis=1, inplace=False)

# format_check = check_date_with_formats(df_inter_for_ds, 'last_watch_dt')
# print("Результаты проверки форматов:")
# for key, value in format_check.items():
#     print(f"{key}: {value}")
     

print(df_inter["item_id"].count())


# df_inter_for_ds["date_w"] = convert_to_datetime (df_inter_for_ds["last_watch_dt"])


5476250


In [431]:
# # делим на трейн и тест(валидация)
# df_inter_for_ds.sort_values(by = "date_w",  ascending=False)
# df_inter_for_ds.drop(columns=["last_watch_dt"],inplace=True)

# days = 7
# last_date = df_inter_for_ds["date_w"].max()

# df_interactions_train = df_inter_for_ds[df_inter_for_ds["date_w"] < last_date - timedelta(days=days)]

# df_interactions_test = df_inter_for_ds[df_inter_for_ds["date_w"] >= last_date - timedelta(days=days)]

In [432]:
def apk(actual, predicted, k=10):
    if len(predicted) > k:
        predicted = predicted[:k]

    score = 0.0
    num_hits = 0.0

    for i, p in enumerate(predicted):
        if p in actual and p not in predicted[:i]:
            num_hits += 1.0
            score += num_hits / (i+1.0)

    if not actual:
        return 0.0

    return score / min(len(actual), k)

In [433]:
def mapk(actual, predicted, k=10):
    return np.mean([apk(a, predicted[u], k) for u, a in actual.items()])

In [434]:
test_size_days=10

In [435]:
from datetime import datetime, timedelta

# Тестовый промежуток времени равен 10 дней
max_date = df_interactions['last_watch_dt'].max()
test_start = max_date - timedelta(days=test_size_days)

In [436]:
df_interactions_train = df_interactions[df_interactions['last_watch_dt'] < test_start]
df_interactions_test = df_interactions[df_interactions['last_watch_dt'] >= test_start]

In [437]:
df_inter.columns

Index(['user_id', 'item_id', 'last_watch_dt', 'watched_pct', 'completed'], dtype='object')

In [438]:
# не обрабатываем холодных пользователей
df_interactions_test = df_interactions_test.loc[df_interactions_test["user_id"].isin(df_interactions_train["user_id"]) & df_interactions_test["item_id"].isin(df_interactions_train["item_id"])]

In [439]:
df_users[['user_id',"age","income"]]

,user_id,age,income
0,973171,age_25_34,income_60_90
1,962099,age_18_24,income_20_40
2,1047345,age_45_54,income_40_60
3,721985,age_45_54,income_20_40
4,704055,age_35_44,income_60_90
...,...,...,...
840191,365945,age_25_34,income_20_40
840192,339025,age_65_inf,income_0_20
840193,983617,age_18_24,income_20_40
840194,251008,NaN,NaN


In [440]:
# ratings = pd.DataFrame()
# ratings[Columns.User] = df_interactions_train['user_id']
# ratings[Columns.Item] = df_interactions_train['item_id']
# ratings[Columns.Weight] = df_interactions_train['watched_pct']
# ratings[Columns.Datetime] = df_interactions_train['last_watch_dt']

In [441]:
from sklearn.preprocessing import LabelEncoder

# Кодируем категориальные признаки (но можно их закодировать прямо при построении датасета)
le_age = LabelEncoder()
le_inc = LabelEncoder()
le_sex = LabelEncoder()
df_users["age_en"] = le_age.fit_transform(df_users["age"])
df_users["income_en"] = le_inc.fit_transform(df_users["income"])
df_users["sex_en"] = le_inc.fit_transform(df_users["sex"])
df_users.head()

,user_id,age,income,sex,kids_flg,age_en,income_en,sex_en
0,973171,age_25_34,income_60_90,М,1,1,4,1
1,962099,age_18_24,income_20_40,М,0,0,2,1
2,1047345,age_45_54,income_40_60,Ж,0,3,3,0
3,721985,age_45_54,income_20_40,Ж,0,3,2,0
4,704055,age_35_44,income_60_90,Ж,0,2,4,0


In [442]:
df_interactions_train.head()

,user_id,item_id,last_watch_dt,total_dur,watched_pct,completed
0,176549,9506,2021-05-11,4250,72.0,1
1,699317,1659,2021-05-29,8317,100.0,1
2,656683,7107,2021-05-09,10,0.0,0
3,864613,7638,2021-07-05,14483,100.0,1
4,964868,9506,2021-04-30,6725,100.0,1


In [443]:
df_interactions_test.head()

,user_id,item_id,last_watch_dt,total_dur,watched_pct,completed
6,1016458,354,2021-08-14,1672,25.0,0
9,203219,13582,2021-08-22,6975,100.0,1
54,200197,9335,2021-08-16,83,2.0,0
64,73446,14488,2021-08-19,6011,100.0,1
84,10010,512,2021-08-15,14,0.0,0


In [444]:
df_interactions_test.rename(columns={'user_id': Columns.User, 'item_id': Columns.Item, 
                                'last_watch_dt': Columns.Datetime, 'watched_pct': Columns.Weight},inplace=True)


df_interactions_train.rename(columns={'user_id': Columns.User, 'item_id': Columns.Item, 
                                'last_watch_dt': Columns.Datetime, 'watched_pct': Columns.Weight},inplace=True)

C:\Users\kanze\AppData\Local\Temp\ipykernel_8788\1242639805.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_interactions_train.rename(columns={'user_id': Columns.User, 'item_id': Columns.Item,


In [445]:
def check_nan (df):
    # Проверка, есть ли вообще NaN значения в DataFrame
    has_any_nans = df.isna().any().any()
    print(f"Есть ли NaN в DataFrame: {has_any_nans}")

    # Столбцы, содержащие хотя бы один NaN
    columns_with_nans = df.columns[df.isna().any()].tolist()
    print(f"Столбцы с NaN: {columns_with_nans}")

    # Количество не-NaN значений
    non_nan_count = df.count()  # По столбцам
    print("Не-NaN значений по столбцам:")
    print(non_nan_count)
    

In [446]:
check_nan (df_interactions_train)

# Удаляем все строки, где есть хотя бы один NaN
df_interactions_train = df_interactions_train.dropna()

check_nan (df_interactions_train)

Есть ли NaN в DataFrame: True
Столбцы с NaN: ['weight']
Не-NaN значений по столбцам:
user_id      4811285
item_id      4811285
datetime     4811285
total_dur    4811285
weight       4810459
completed    4811285
dtype: int64
Есть ли NaN в DataFrame: False
Столбцы с NaN: []
Не-NaN значений по столбцам:
user_id      4810459
item_id      4810459
datetime     4810459
total_dur    4810459
weight       4810459
completed    4810459
dtype: int64


In [447]:
check_nan (df_interactions_test)

# Удаляем все строки, где есть хотя бы один NaN
df_interactions_test = df_interactions_test.dropna()

check_nan (df_interactions_test)

Есть ли NaN в DataFrame: True
Столбцы с NaN: ['weight']
Не-NaN значений по столбцам:
user_id      435153
item_id      435153
datetime     435153
total_dur    435153
weight       435151
completed    435153
dtype: int64
Есть ли NaN в DataFrame: False
Столбцы с NaN: []
Не-NaN значений по столбцам:
user_id      435151
item_id      435151
datetime     435151
total_dur    435151
weight       435151
completed    435151
dtype: int64


In [448]:
# #user_features_df= df_users[["user_id","income","age","sex"]]

 

dc = {
    Columns.User: [],  # Используем константу Columns.User везде
    "feature": [],
    "value": []
}

for index, row in df_users.iterrows():  
    dc[Columns.User].append(row["user_id"])
    dc["feature"].append("income")
    dc["value"].append(row["income_en"])

    dc[Columns.User].append(row["user_id"])  # Исправлено!
    dc["feature"].append("age")
    dc["value"].append(row["age_en"])

    dc[Columns.User].append(row["user_id"])  # Исправлено!
    dc["feature"].append("sex")
    dc["value"].append(row["sex_en"])

user_features_df = pd.DataFrame(dc)

In [449]:
# from rectools import  dataset
# # Создаем Dataset из rectools
# dataset = dataset.Dataset.construct(ratings)

# users_uniq = ratings[Columns.User].unique()

In [450]:
from rectools import  dataset 
# Prepare a dataset to build a model
rating = dataset.Dataset.construct(interactions_df = df_interactions_train, 
                                   user_features_df=user_features_df, 
                                   #item_features_df=df_items
                                   make_dense_user_features=False
                                   )

In [451]:
from rectools.models import LightFMWrapperModel
from lightfm import LightFM

model = LightFMWrapperModel(
        # внутри модели указываем параметр no_components
        # это размезность эмбеддингов, которые выучит модель
        model=LightFM(no_components = 10,  random_state=42),
        )


In [452]:
# from rectools import  dataset
# # Создаем Dataset из rectools
# dataset = dataset.Dataset.construct(ratings)

# users_uniq = ratings[Columns.User].unique()

In [453]:
rating_val = dataset.Dataset.construct(interactions_df = df_interactions_test, 
                                   user_features_df=user_features_df, 
                                   #item_features_df=df_items
                                   make_dense_user_features=False
                                   )

In [454]:
# users_unique = df_inter_for_test_cleaned[Columns.User].unique()
# model.fit(rating_val)
# recos = model.recommend(
#     users=users_unique,
#     dataset=rating_val,
#     k= 10,
#     filter_viewed= True
   
# )

In [455]:
users_uniq = df_interactions_test[Columns.User].unique()

In [456]:
# users_unique = df_inter_for_test_cleaned[Columns.User].unique()



In [457]:
model.fit(rating_val)

In [458]:
recom = model.recommend(
    users=users_uniq,
    dataset=rating_val,
    k= 10,
    filter_viewed= True
   
)

In [459]:
recom

,user_id,item_id,score,rank
0,1016458,15297,20.916502,1
1,1016458,9728,20.656118,2
2,1016458,10440,20.498997,3
3,1016458,13865,20.168795,4
4,1016458,8636,20.077984,5
...,...,...,...,...
1378415,442859,4495,21.938150,6
1378416,442859,11118,21.875683,7
1378417,442859,3784,21.806181,8
1378418,442859,7793,21.739054,9


In [460]:
df_recom = recom.groupby(Columns.User)[Columns.Item].unique()

In [461]:
df_recom.columns = Columns.UserItem

In [462]:
df_recom

user_id
3          [15297, 9728, 10440, 11118, 13865, 7793, 14488...
9          [15297, 9728, 10440, 13865, 8636, 4495, 11118,...
14         [15297, 10440, 9728, 11118, 13865, 14488, 7793...
17         [15297, 9728, 10440, 13865, 8636, 4495, 11118,...
21         [15297, 9728, 10440, 13865, 8636, 4495, 11118,...
                                 ...                        
1097515    [15297, 9728, 11118, 13865, 7793, 14488, 8636,...
1097521    [15297, 9728, 10440, 13865, 8636, 4495, 11118,...
1097525    [15297, 9728, 10440, 13865, 8636, 4495, 3784, ...
1097527    [15297, 9728, 13865, 8636, 10440, 4495, 3784, ...
1097544    [15297, 9728, 10440, 13865, 8636, 4495, 11118,...
Name: item_id, Length: 137842, dtype: object

In [463]:
# Создаем словарь для маппинга item_id в названия
item_id_to_title = df_items.set_index('item_id')['title'].to_dict()

# Функция для преобразования списка ID в названия
def map_ids_to_titles(item_ids, mapping_dict):
    return [mapping_dict.get(item_id, f"Unknown (ID: {item_id})") for item_id in item_ids]

# Применяем маппинг к рекомендациям
df_recom_with_titles = df_recom.apply(lambda x: map_ids_to_titles(x, item_id_to_title))

# Выводим результат с названиями фильмов
print("Рекомендации с названиями фильмов:")
for user_id, movie_titles in df_recom_with_titles.head(100).items():
    print(f"Пользователь {user_id}:")
    for i, title in enumerate(movie_titles, 1):
        print(f"  {i}. {title}")
    print()

Рекомендации с названиями фильмов:
Пользователь 3:
  1. Клиника счастья
  2. Гнев человеческий
  3. Хрустальный
  4. Райская бухта
  5. Девятаев
  6. Радиовспышка
  7. Мастер меча
  8. Белый снег
  9. Маленький воин
  10. ДУХLESS

Пользователь 9:
  1. Клиника счастья
  2. Гнев человеческий
  3. Хрустальный
  4. Девятаев
  5. Белый снег
  6. Пальмира
  7. Райская бухта
  8. Маленький воин
  9. Радиовспышка
  10. Мастер меча

Пользователь 14:
  1. Клиника счастья
  2. Хрустальный
  3. Гнев человеческий
  4. Райская бухта
  5. Девятаев
  6. Мастер меча
  7. Радиовспышка
  8. Белый снег
  9. Маленький воин
  10. Апгрейд

Пользователь 17:
  1. Клиника счастья
  2. Гнев человеческий
  3. Хрустальный
  4. Девятаев
  5. Белый снег
  6. Пальмира
  7. Райская бухта
  8. Маленький воин
  9. Радиовспышка
  10. Суперсемейка 2

Пользователь 21:
  1. Клиника счастья
  2. Гнев человеческий
  3. Хрустальный
  4. Девятаев
  5. Белый снег
  6. Пальмира
  7. Райская бухта
  8. Маленький воин
  9. Радиовсп

In [464]:
# map10 = mapk(df_recom, predicted, k=10)
# map10

In [470]:

actual_dict = df_interactions_test.groupby(Columns.User)[Columns.Item].apply(list).to_dict()

predicted_dict = df_recom.to_dict()

mapk10_score = mapk(actual_dict, predicted_dict, k=10)


In [471]:
mapk10_score

0.0

In [466]:
# import pandas as pd
# from sklearn.metrics import recall_score

# d = {"user_id": [], "recos": [], 'actual': []} 
# recall_scores = []

# for u in users_unique:
#     d["user_id"].append(u)
    
#     # Рекомендации
#     r = (recos[Columns.Item][recos[Columns.User] == u]).tolist()
#     d["recos"].append(r)
    
#     # Фактические взаимодействия
#     a = (df_inter_for_test_cleaned[Columns.Item][df_inter_for_test_cleaned[Columns.User] == u]).tolist()
#     d["actual"].append(a)

#     print(f"user {u}:\n rec {r} \n act {a}")

#     # Вычисление recall для текущего пользователя
#     if len(a) > 0:  # Проверяем, что есть фактические элементы
#         # Создаем бинарные векторы для вычисления recall
#         all_items = list(set(r + a))
#         y_true = [1 if item in a else 0 for item in all_items]
#         y_pred = [1 if item in r else 0 for item in all_items]
        
#         user_recall = recall_score(y_true, y_pred, zero_division=0)
#         recall_scores.append(user_recall)
#     else:
#         recall_scores.append(0)  # Если нет фактических элементов, recall = 0

# # Средний recall по всем пользователям
# mean_recall = np.mean(recall_scores) if recall_scores else 0
# print(f"Mean Recall: {mean_recall}")

In [467]:
# import metrics

# d = {"user_id":[], "recos": [], 'actual':[]} 
# recall = 0

# for u in users_unique:
#     d["user_id"].append(u)
#     r = (recos[Columns.Item][recos[Columns.User] == u]).to_list()
#     d["recos"].append(r)
#     a = (df_inter_for_test_cleaned[Columns.Item][df_inter_for_test_cleaned[Columns.User] == u]).to_list()
#     d["actual"].append(a)

#     print(f"user {u}:\n rec {r} \n act {a}")

#     recall += metrics.recall(r,a)

# recall = recall/len(users_unique)

# print(recall)